In [3]:
import QuantLib as ql
import pandas as pd


In [2]:
pip install QuantLib

     --------------------------------------- 12.6/12.6 MB 11.3 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [4]:
ql.IborIndex('MyIndex', ql.Period('6m'), 2, ql.EURCurrency(), ql.TARGET(), ql.ModifiedFollowing, True, ql.Actual360())
ql.Libor('MyIndex', ql.Period('6M'), 2, ql.USDCurrency(), ql.TARGET(), ql.Actual360())
ql.Euribor(ql.Period('6M'))
ql.USDLibor(ql.Period('6M'))
ql.Euribor6M()

<QuantLib.QuantLib.Euribor6M; proxy of <Swig Object of type 'ext::shared_ptr< Euribor6M > *' at 0x000002868E3173F0> >

In [5]:
name = 'CNYRepo7D'
fixingDays = 1
currency = ql.CNYCurrency()
calendar = ql.China()
dayCounter = ql.Actual365Fixed()
overnight_index = ql.OvernightIndex(name, fixingDays, currency, calendar, dayCounter)

In [6]:
today = ql.Date(27, ql.October, 2024)
ql.Settings.instance().evaluationDate = today

bps = 1e-4

In [7]:
sample_rates = pd.DataFrame(
    [
        (ql.Date(27, 10, 2024), 0.0229, 0.0893, -0.5490, -0.4869),
        (ql.Date(27, 1, 2025), 0.0645, 0.1059, -0.5584, -0.5057),
        (ql.Date(27, 4, 2025), 0.0414, 0.1602, -0.5480, -0.5236),
        (ql.Date(27, 10, 2025), 0.1630, 0.2601, -0.5656, -0.5030),
        (ql.Date(27, 10, 2026), 0.4639, 0.6281, -0.4365, -0.3468),
        (ql.Date(27, 10, 2027), 0.7187, 0.9270, -0.3500, -0.2490),
        (ql.Date(27, 10, 2028), 0.9056, 1.1257, -0.3041, -0.1590),
        (ql.Date(27, 10, 2029), 1.0673, 1.2821, -0.2340, -0.0732),
        (ql.Date(27, 10, 2030), 1.1615, 1.3978, -0.1690, -0.0331),
        (ql.Date(27, 10, 2031), 1.2326, 1.4643, -0.1041, 0.0346),
        (ql.Date(27, 10, 2032), 1.3050, 1.5589, -0.0070, 0.1263),
        (ql.Date(27, 10, 2033), 1.3584, 1.5986, 0.0272, 0.1832),
        (ql.Date(27, 10, 2034), 1.4023, 1.6488, 0.0744, 0.2599),
        (ql.Date(27, 10, 2035), 1.5657, 1.8136, 0.3011, 0.4406),
        (ql.Date(27, 10, 2040), 1.6191, 1.8749, 0.3882, 0.5331),
        (ql.Date(27, 10, 2046), 1.6199, 1.8701, 0.3762, 0.5225),
        (ql.Date(27, 10, 2051), 1.6208, 1.8496, 0.3401, 0.4926),
    ],
    columns=["date", "SOFR", "USDLibor3M", "EUR-USD-discount", "Euribor3M"],
)

In [8]:
def sample_curve(tag):
    curve = ql.ZeroCurve(
        sample_rates["date"], sample_rates[tag] / 100, ql.Actual365Fixed()
    )
    return ql.YieldTermStructureHandle(curve)

In [11]:
notional = 1_000_000
fx_0 = 0.85
spread = 50.0 * bps

In [15]:
euribor3M_curve = sample_curve("Euribor3M")
euribor3M = ql.Euribor(ql.Period(3, ql.Months), euribor3M_curve)

usdlibor3M_curve = sample_curve("USDLibor3M")
usdlibor3M = ql.USDLibor(ql.Period(3, ql.Months), usdlibor3M_curve)

In [16]:
calendar = ql.UnitedStates(ql.UnitedStates.FederalReserve)
start_date = calendar.advance(today, ql.Period(2, ql.Days))
end_date = calendar.advance(start_date, ql.Period(5, ql.Years))
tenor = ql.Period(3, ql.Months)
rule = ql.DateGeneration.Forward
convention = ql.Following
end_of_month = False

schedule = ql.Schedule(
    start_date,
    end_date,
    tenor,
    calendar,
    convention,
    convention,
    rule,
    end_of_month,
)

In [17]:
usd_leg = (
    (ql.SimpleCashFlow(-notional, schedule[0]),)
    + ql.IborLeg(
        nominals=[notional],
        schedule=schedule,
        index=usdlibor3M,
        spreads=[spread],
    )
    + (ql.SimpleCashFlow(notional, schedule[-1]),)
)

In [18]:
sofr_curve = sample_curve("SOFR")
usd_npv = ql.CashFlows.npv(usd_leg, sofr_curve, True)
usd_npv

35427.65532574046

NameError: name 'eur_leg' is not defined

In [22]:
eur_leg = (
    (ql.SimpleCashFlow(-notional * fx_0, schedule[0]),)
    + ql.IborLeg(nominals=[notional * fx_0], schedule=schedule, index=euribor3M)
    + (ql.SimpleCashFlow(notional * fx_0, schedule[-1]),)
)

In [23]:
eurusd_curve = sample_curve("EUR-USD-discount")
eur_npv = ql.CashFlows.npv(eur_leg, eurusd_curve, True)
eur_npv

6908.723028828739

In [24]:
ql.CashFlows.npv(eur_leg, eurusd_curve, True) / fx_0

8127.90944568087

In [25]:
def cashflow_data(leg):
    data = []
    for cf in sorted(leg, key=lambda c: c.date()):
        coupon = ql.as_floating_rate_coupon(cf)
        if coupon is None:
            data.append((cf.date(), None, None, cf.amount()))
        else:
            data.append(
                (
                    coupon.date(),
                    coupon.nominal(),
                    coupon.rate(),
                    coupon.amount(),
                )
            )
    return pd.DataFrame(
        data, columns=["date", "nominal", "rate", "amount"]
    ).style.format({"amount": "{:.2f}", "nominal": "{:.2f}", "rate": "{:.2%}"})

In [26]:
cashflow_data(usd_leg)

,date,nominal,rate,amount
0,"October 29th, 2021",nan,nan%,-1000000.00
1,"January 31st, 2022",1000000.00,0.61%,1585.56
2,"April 29th, 2022",1000000.00,0.72%,1750.57
3,"July 29th, 2022",1000000.00,0.81%,2040.59
4,"October 31st, 2022",1000000.00,0.91%,2386.92
5,"January 30th, 2023",1000000.00,1.22%,3080.34
6,"May 1st, 2023",1000000.00,1.40%,3541.30
7,"July 31st, 2023",1000000.00,1.58%,3999.86
8,"October 30th, 2023",1000000.00,1.76%,4444.64
9,"January 29th, 2024",1000000.00,1.79%,4518.96


In [27]:
cashflow_data(eur_leg)

,date,nominal,rate,amount
0,"October 29th, 2021",nan,nan%,-850000.00
1,"January 31st, 2022",850000.00,-0.50%,-1108.91
2,"April 29th, 2022",850000.00,-0.53%,-1109.57
3,"July 29th, 2022",850000.00,-0.49%,-1042.88
4,"October 31st, 2022",850000.00,-0.46%,-1020.88
5,"January 30th, 2023",850000.00,-0.30%,-644.90
6,"May 1st, 2023",850000.00,-0.22%,-479.05
7,"July 31st, 2023",850000.00,-0.15%,-314.08
8,"October 30th, 2023",850000.00,-0.07%,-158.20
9,"January 29th, 2024",850000.00,-0.12%,-266.58
